I reviewed the notebook you uploaded and integrated its **simple conceptual explanation** with your **full technical documentation** to produce a **single comprehensive documentation file**. The result keeps three layers of explanation:

1. **Intuitive explanation (from the notebook)**
2. **Technical explanation (from your document)**
3. **Operational usage instructions (scripts, experiments, evaluation)**

This makes the document suitable for:

* **reports**
* **research documentation**
* **project repositories**
* **student explanations**

Below is the **final integrated documentation file**.

---

# CA-DRL / DDQN Experiment Documentation

## Training Data, Testing Data, State Representation, and Action Space

This document explains the key experimental concepts used in the CA-DRL and DDQN satellite IoT scheduling experiments.

The goal is to make the system understandable from both:

* a **simple conceptual perspective**, and
* a **technical reinforcement learning implementation perspective**.

The document covers:

* Reinforcement learning **training and evaluation data**
* The **50-dimensional state representation**
* The **21-action decision space**
* How to use the **predict_from_state evaluation script**
* Example **state inputs for testing trained models**

---

# 1. Conceptual Overview (Simple Intuition)

To understand the problem intuitively, imagine the following system:

A **satellite communication system** with:

* **14 communication channels**
* Many **IoT sensors** trying to transmit data
* A **waiting queue** of sensors that cannot transmit yet

At any time step:

1. Some channels are **already busy** transmitting data.
2. Some sensors are **waiting in a queue** to transmit.
3. The system must decide **which sensor to schedule next**.

An **AI agent** observes the system state and chooses one action:

* Schedule one sensor from the queue, or
* Do nothing for this step.

The goal of the agent is to **minimize delay and improve throughput**.

---

# 2. Training Data vs Testing / Evaluation Data

Unlike supervised learning problems where a fixed dataset exists, this project uses **reinforcement learning (RL)**.

In reinforcement learning:

* The agent **generates its own data** by interacting with the environment.

The environment simulates a **satellite IoT communication network**.

---

## Training Process

At each time step the agent performs the following sequence:

1. The agent observes the **current state** $s_t$

2. The agent selects an **action** $a_t$

3. The environment returns a **reward** $r_t$

4. The environment transitions to a **new state** $s_{t+1}$

Each interaction produces a **transition tuple**:

$$
(s_t, a_t, r_t, s_{t+1}, done)
$$

Where:

* $s_t$ = current state
* $a_t$ = chosen action
* $r_t$ = reward
* $s_{t+1}$ = next state
* `done` = episode termination indicator

These transitions are stored in a **Replay Buffer**.

During training:

* The neural network samples **mini-batches of transitions**
* The Q-network parameters are updated.

Therefore:

**Training data consists of all transitions collected while the agent interacts with the environment during training.**

---

## Testing / Evaluation Process

Evaluation occurs **after training finishes**.

During evaluation:

* The trained model is **frozen**
* No gradient updates occur
* The agent runs with its **learned policy**

The model is tested across many **new environment episodes**.

Performance metrics are recorded such as:

* Average queue delay
* Average reward
* System throughput

These evaluation episodes serve as the **testing dataset**, because:

* They are generated **after training**
* They are used **only for performance measurement**
* They **do not affect the model parameters**

---

# 3. State Representation (50-Dimensional Vector)

The system state representation follows the design used in the CA-DRL research paper.

The state at time (t) is defined as:

$$
S_t =
[ tasks_1...tasks_T , bw_1...bw_N , q_1...q_N ]
$$

Where:

* $T = 10$ → number of future time steps observed
* $N = 20$ → maximum queue size

Therefore:

$$
state_dim = T + 2N = 10 + 2(20) = 50
$$

Each state is a **vector of 50 numbers**.

---

# 4. Simple View of the 50 Numbers

Each state contains **three parts**.

### 1. Future Channel Usage (10 numbers)

`tasks_1 … tasks_10`

These values describe **how busy the channels will be in the near future**.

Example:

```
[2,1,1,0,0,0,0,0,0,0]
```

Meaning:

* Next step → 2 active transmissions
* Two steps later → 1 transmission
* Three steps later → 1 transmission
* After that → channels become free

This allows the agent to **predict congestion**.

---

### 2. Sensor Data Size (20 numbers)

`bw_1 … bw_20`

These represent the **data size of sensors waiting in the queue**.

Properties:

* Ordered by **queue position**
* `bw_1` = first sensor in queue
* If queue < 20 sensors → remaining values are **0**

Typical range:

```
100 – 1000 bits
```

---

### 3. Sensor Transmission Duration (20 numbers)

`q_1 … q_20`

These represent the **transmission duration** of queued sensors.

Properties:

* Ordered by queue position
* If queue has fewer than 20 sensors → remaining values are **0**

Typical values:

```
1 – 15 time steps
```

---

## State Layout Summary

| Index Range | Meaning                       |
| ----------- | ----------------------------- |
| 0 – 9       | Future active transmissions   |
| 10 – 29     | Sensor data sizes             |
| 30 – 49     | Sensor transmission durations |

---

# 5. Action Space (21 Actions)

At each time step the agent must choose **one action**.

The environment defines:

[
action_dim = N + 1 = 21
]

Where:

```
N = 20 queue slots
```

---

## Meaning of Each Action

### Actions 0 – 19

Each action corresponds to **one queue position**.

Example:

| Action | Meaning                              |
| ------ | ------------------------------------ |
| 0      | Schedule sensor at queue position 0  |
| 1      | Schedule sensor at queue position 1  |
| ...    | ...                                  |
| 19     | Schedule sensor at queue position 19 |

If:

* a sensor exists in that queue position **and**
* a channel is available

Then the sensor begins transmission and is **removed from the queue**.

If no sensor exists at that position, the action has **no effect**.

---

### Action 20 (No-Operation)

Action 20 represents **doing nothing**.

This means:

* The agent chooses **not to schedule any sensor** in this step.

This option is important because sometimes it is **better to wait** for a channel to free up.

---

## Policy Selection

The neural network outputs **21 Q-values**.

The chosen action is:

$$
a^* = argmax_a Q(s,a)
$$

Meaning the agent selects the action with the **highest predicted value**.

---

# 6. Independent Prediction Script

The project includes a script:

```
evaluation/predict_from_state.py
```

This script allows testing a trained model on **any single state vector**.

The script outputs:

* The **chosen action**
* The **Q-values for all 21 actions**

This makes it possible to analyze model behavior **outside full simulations**.

---

# 7. Running the Prediction Script

## Using CA-DRL

```
cd ca-ddqn
python -m evaluation.predict_from_state --agent cadrl
```

---

## Using DDQN

```
cd ca-ddqn
python -m evaluation.predict_from_state --agent ddqn
```

---

# 8. Providing Custom State Input

If no state is provided:

The script:

1. Creates the environment
2. Calls

```
env.reset()
```

3. Generates a valid state vector.

---

If a state file is provided:

The script loads the state from:

* JSON file containing **50 numbers**, or
* NumPy `.npy` file containing a **1D array of length 50**

The script verifies that the state dimension equals **50** before running the model.

---

# 9. Example State Files

Two example states are provided.

---

## Light System Load

```
evaluation/sample_state_sparse.json
```

Characteristics:

* Few active transmissions
* Small queue
* Small sensor data

Useful for **sanity testing**.

---

## Heavy System Load

```
evaluation/sample_state_heavy_load.json
```

Characteristics:

* Many active transmissions
* Large queue
* Larger sensor data sizes

Useful for analyzing **model behavior under congestion**.

---

# 10. Example Commands

## CA-DRL

```
cd ca-ddqn

python -m evaluation.predict_from_state \
--agent cadrl \
--state-file evaluation/sample_state_sparse.json

python -m evaluation.predict_from_state \
--agent cadrl \
--state-file evaluation/sample_state_heavy_load.json
```

---

## DDQN

```
cd ca-ddqn

python -m evaluation.predict_from_state \
--agent ddqn \
--state-file evaluation/sample_state_sparse.json

python -m evaluation.predict_from_state \
--agent ddqn \
--state-file evaluation/sample_state_heavy_load.json
```

---

# 11. Purpose of These Tests

These state-based tests allow researchers to:

* Inspect **model decisions**
* Compare **CA-DRL vs DDQN behavior**
* Evaluate **scheduling strategies**
* Understand how models respond to **different network conditions**

without running full training or long simulation episodes.

---
